# 8.5.循环神经网络的从零开始实现

本节将根据 [8.4节](./08.04_rnn.ipynb)中的描述，从头开始基于循环神经网络实现字符级语言模型。这样的模型将在H.G.Wells的时光机器数据集上训练。和前面 [8.3节](./08.03_language_models_and_dataset.ipynb)中介绍过的一样，我们先读取数据集。


## 环境配置


In [1]:
%pip install pypto==0.2.0 torch torch_npu matplotlib

In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor with internal format")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import torch_npu
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

In [3]:
from src.utils import load_data_time_machine

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l


batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
    </pre>
  </div>
</details>


## 8.5.1.独热编码

回想一下，在`train_iter`中，每个词元都表示为一个数字索引，将这些索引直接输入神经网络可能会使学习变得困难。我们通常将每个词元表示为更具表现力的特征向量。最简单的表示称为*独热编码*（one-hot encoding），它在 [3.4.1节](../03_pypto_linear_networks/03.04_softmax_regression.ipynb)中介绍过。

简言之，将每个索引映射为相互不同的单位向量：假设词表中不同词元的数目为$N$（即`len(vocab)`），词元索引的范围为$0$到$N-1$。如果词元的索引是整数$i$，那么我们将创建一个长度为$N$的全$0$向量，并将第$i$处的元素设置为$1$。此向量是原始词元的一个独热向量。索引为$0$和$2$的独热向量如下所示：


我们每次采样的小批量数据形状是二维张量：（批量大小，时间步数）。`one_hot`函数将这样一个小批量数据转换成三维张量，张量的最后一个维度等于词表大小（`len(vocab)`）。我们经常转换输入的维度，以便获得形状为（时间步数，批量大小，词表大小）的输出。这将使我们能够更方便地通过最外层的维度，一步一步地更新小批量数据的隐状态。


In [4]:
F.one_hot(torch.tensor([0, 2]), len(vocab))

# 批量数据的独热编码：形状从 (batch, steps) 变为 (steps, batch, vocab)
X = torch.arange(10).reshape((2, 5))
F.one_hot(X.T, len(vocab)).shape

torch.Size([5, 2, 28])

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
F.one_hot(torch.tensor([0, 2]), len(vocab))
X = torch.arange(10).reshape((2, 5))
F.one_hot(X.T, 28).shape
</pre>
  </div>
</details>


## 8.5.2.初始化模型参数

接下来，我们**初始化循环神经网络模型的模型参数**。隐藏单元数`num_hiddens`是一个可调的超参数。当训练语言模型时，输入和输出来自相同的词表。因此，它们具有相同的维度，即词表的大小。


In [5]:
def get_params(vocab_size, num_hiddens, device):
    """初始化循环神经网络模型参数。"""
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01

    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    # 隐藏层参数
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params
    </pre>

  </div>
</details>


## 8.5.3.循环神经网络模型

为了定义循环神经网络模型，我们首先需要**一个`init_rnn_state`函数在初始化时返回隐状态**。这个函数的返回是一个张量，张量全用0填充，形状为（批量大小，隐藏单元数）。在后面的章节中我们将会遇到隐状态包含多个变量的情况，而使用元组可以更容易地处理些。


In [6]:
def init_rnn_state(batch_size, num_hiddens, device):
    """初始化 RNN 隐状态为零张量。"""
    return (torch.zeros((batch_size, num_hiddens), device=device), )

[8.4 节](./08.04_rnn.ipynb)的 RNN 公式涉及四个 PyPTO 算子：matmul、bias_add、add（逐元素加法）和 tanh 激活。其中 **matmul 与 bias_add 已在 [4.2 节](../04_pypto_multilayer_perceptrons/04.02_mlp_scratch.ipynb)中完整实现**，从 `src.pypto_ops` 直接导入即可复用。

add 与 tanh 虽也已收录在 `src/pypto_ops.py` 中，但为保持教学完整性，**本节将内联展示其完整的Kernel + 工厂函数链条**，帮助读者看清 RNN 中每个算子在 NPU 上的完整工作流。后续[8.6 节](./08.06_rnn_concise.ipynb)将从共享模块直接导入，不再重复。


In [7]:
from src.pypto_ops import PyPTOMatmul, PyPTOBiasAdd, loss_fn
# RNN 专用的 add + tanh 算子（matmul/bias_add 见 [4.2 节]）
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def add_fwd_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    c: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    c.move(pypto.add(a, b))

@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def tanh_fwd_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    two_a = pypto.mul(a, 2.0)
    s = pypto.sigmoid(two_a)
    two_s = pypto.mul(s, 2.0)
    b.move(pypto.sub(two_s, 1.0))

@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def tanh_bwd_kernel(
    b: pypto.Tensor([], pypto.DT_FP32),
    grad_b: pypto.Tensor([], pypto.DT_FP32),
    grad_a: pypto.Tensor([], pypto.DT_FP32),
):
    pypto.set_vec_tile_shapes(128, 128)
    b_sq = pypto.mul(b, b)
    one_minus_b2 = pypto.neg(pypto.sub(b_sq, 1.0))
    grad_a.move(pypto.mul(grad_b, one_minus_b2))

# 将内核包装为 torch.autograd.Function
def make_pypto_add(fwd_kernel):
    class PyPTOAddImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, a, b):
            ctx.save_for_backward(a, b)
            c = torch.empty_like(a)
            fwd_kernel(a, b, c)
            return c
        @staticmethod
        def backward(ctx, grad_c):
            grad_a = grad_c if ctx.needs_input_grad[0] else None
            grad_b = grad_c if ctx.needs_input_grad[1] else None
            return grad_a, grad_b
    return PyPTOAddImpl

def make_pypto_tanh(fwd_kernel, bwd_kernel):
    class PyPTOTanhImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, a):
            b = torch.empty_like(a)
            fwd_kernel(a, b)
            ctx.save_for_backward(b)
            return b
        @staticmethod
        def backward(ctx, grad_b):
            (b,) = ctx.saved_tensors
            grad_a = torch.empty_like(b) if ctx.needs_input_grad[0] else None
            if grad_a is not None:
                bwd_kernel(b.contiguous(), grad_b.contiguous(), grad_a)
            return grad_a
    return PyPTOTanhImpl

PyPTOAdd = make_pypto_add(add_fwd_kernel)
PyPTOTanh = make_pypto_tanh(tanh_fwd_kernel, tanh_bwd_kernel)


In [8]:
def rnn_forward_pypto(inputs, state, params):
    """使用 PyPTO 算子在所有时间步上执行 RNN 前向传播。"""
    W_xh, W_hh, b_h, W_hq, b_q = params
    (H,) = state
    outputs = []
    for X in inputs:
        XW = PyPTOMatmul.apply(X, W_xh)
        HW = PyPTOMatmul.apply(H, W_hh)
        summed = PyPTOAdd.apply(XW, HW)
        biased = PyPTOBiasAdd.apply(summed, b_h)
        H = PyPTOTanh.apply(biased)
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

In [9]:
class PyPTORNNModelScratch:
    """PyPTO 从零实现的 RNN 模型。"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params_fn=None, init_state_fn=None, forward_fn=None):
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.device = device
        self.params = (get_params_fn or get_params)(vocab_size, num_hiddens, device)
        self.init_state = init_state_fn or init_rnn_state
        self.forward_fn = forward_fn or rnn_forward_pypto

    def __call__(self, X, state):
        X = torch.nn.functional.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device=None):
        if device is None:
            device = self.device
        return self.init_state(batch_size, self.num_hiddens, device)

In [10]:
num_hiddens = 512
net = PyPTORNNModelScratch(len(vocab), num_hiddens, device)
X = torch.arange(10).reshape((2, 5)).to(device)
state = net.begin_state(X.shape[0], device)
Y, new_state = net(X, state)
Y.shape, len(new_state), new_state[0].shape

(torch.Size([10, 28]), 1, torch.Size([2, 512]))

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>add 算子</b>：前向执行 <code>pypto.add(a, b)</code> 同形状逐元素加法；反向为恒等映射——<code>grad_a = grad_c</code>、<code>grad_b = grad_c</code>，直接在 backward 中复用上游梯度，无需 NPU 内核。</li>
      <li style="margin: 0 0 8px 0;"><b>tanh 算子</b>：pypto 0.2.0 无内置 <code>pypto.tanh</code> API，用公式 <code>tanh(x) = (e<sup>2x</sup> - 1) / (e<sup>2x</sup> + 1)</code> 组合实现。反向利用 <code>d(tanh)/dx = 1 - tanh²(x)</code>，前向输出 <code>b</code>（即 tanh 值）直接作为反向输入，省去重算 <code>e<sup>2x</sup></code>。</li>
      <li style="margin: 0 0 8px 0;"><b>工厂模式</b>：<code>make_pypto_add</code> 和 <code>make_pypto_tanh</code> 沿用第4章工厂模式——接收 kernel 返回 <code>torch.autograd.Function</code>，通过 <code>.apply()</code> 接入 PyTorch 自动微分。</li>
      <li style="margin: 0 0 8px 0;">这些算子已收录在 <code>src/pypto_ops.py</code> 中，<b>后续章节可直接导入</b>（见 <a href="./08.06_rnn_concise.ipynb">8.6 节</a>）。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def init_rnn_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device), )
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def rnn(inputs, state, params):
    # inputs的形状：(时间步数量，批量大小，词表大小)
    W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    # X的形状：(批量大小，词表大小)
    for X in inputs:
        H = torch.tanh(torch.mm(X, W_xh) + torch.mm(H, W_hh) + b_h)
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class RNNModelScratch:
    """从零开始实现的循环神经网络模型"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_hiddens = 512
net = RNNModelScratch(len(vocab), num_hiddens, d2l.try_gpu(), get_params,
                      init_rnn_state, rnn)
state = net.begin_state(X.shape[0], d2l.try_gpu())
Y, new_state = net(X.to(d2l.try_gpu()), state)
Y.shape, len(new_state), new_state[0].shape
</pre>
  </div>
</details>


## 8.5.4.预测

让我们**首先定义预测函数来生成`prefix`之后的新字符**，其中的`prefix`是一个用户提供的包含多个字符的字符串。在循环遍历`prefix`中的开始字符时，我们不断地将隐状态传递到下一个时间步，但是不生成任何输出。这被称为*预热*（warm-up）期，因为在此期间模型会自我更新（例如，更新隐状态），但不会进行预测。预热期结束后，隐状态的值通常比刚开始的初始值更适合预测，从而预测字符并输出它们。

In [11]:
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在 prefix 后生成新字符。"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return "".join([vocab.idx_to_token[i] for i in outputs])

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在prefix后面生成新字符"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:  # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):  # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
predict_ch8('time traveller ', 10, net, vocab, d2l.try_gpu())
</pre>
  </div>
</details>


## 8.5.5.梯度裁剪

对于长度为$T$的序列，我们在迭代中计算这$T$个时间步上的梯度，将会在反向传播过程中产生长度为$\mathcal{O}(T)$的矩阵乘法链。如 [4.8节](../04_pypto_multilayer_perceptrons/04.08_numerical_stability_and_init.ipynb)所述，当$T$较大时，它可能导致数值不稳定，例如可能导致梯度爆炸或梯度消失。因此，循环神经网络模型往往需要额外的方式来支持稳定训练。

一般来说，当解决优化问题时，我们对模型参数采用更新步骤。假定在向量形式的$\mathbf{x}$中，或者在小批量数据的负梯度$\mathbf{g}$方向上。例如，使用$\eta > 0$作为学习率时，在一次迭代中，我们将$\mathbf{x}$更新为$\mathbf{x} - \eta \mathbf{g}$。如果我们进一步假设目标函数$f$表现良好，即函数$f$在常数$L$下是*利普希茨连续的*（Lipschitz continuous）。也就是说，对于任意$\mathbf{x}$和$\mathbf{y}$我们有：

$$\tag{8.5.1}|f(\mathbf{x}) - f(\mathbf{y})| \leq L \|\mathbf{x} - \mathbf{y}\|.$$

在这种情况下，我们可以安全地假设：如果我们通过$\eta \mathbf{g}$更新参数向量，则

$$\tag{8.5.2}|f(\mathbf{x}) - f(\mathbf{x} - \eta\mathbf{g})| \leq L \eta\|\mathbf{g}\|,$$

这意味着我们不会观察到超过$L \eta \|\mathbf{g}\|$的变化。这既是坏事也是好事。坏的方面，它限制了取得进展的速度；好的方面，它限制了事情变糟的程度，尤其当我们朝着错误的方向前进时。

有时梯度可能很大，从而优化算法可能无法收敛。我们可以通过降低$\eta$的学习率来解决这个问题。但是如果我们很少得到大的梯度呢？在这种情况下，这种做法似乎毫无道理。一个流行的替代方案是通过将梯度$\mathbf{g}$投影回给定半径（例如$\theta$）的球来裁剪梯度$\mathbf{g}$。如下式：

$$\tag{8.5.3}\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}.$$

通过这样做，我们知道梯度范数永远不会超过$\theta$，并且更新后的梯度完全与$\mathbf{g}$的原始方向对齐。它还有一个值得拥有的副作用，即限制任何给定的小批量数据（以及其中任何给定的样本）对参数向量的影响，这赋予了模型一定程度的稳定性。梯度裁剪提供了一个快速修复梯度爆炸的方法，虽然它并不能完全解决问题，但它是众多有效的技术之一。

下面我们定义一个函数来裁剪模型的梯度，模型是从零开始实现的模型或由高级API构建的模型。我们在此计算了所有模型参数的梯度的范数。


In [12]:
import math

def grad_clipping(net, theta):
    """裁剪梯度。"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def grad_clipping(net, theta):
    """裁剪梯度"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm
    </pre>

  </div>
</details>


## 8.5.6.训练

在训练模型之前，让我们**定义一个函数在一个迭代周期内训练模型**。它与我们训练 [3.6节](../03_pypto_linear_networks/03.06_softmax_regression_scratch.ipynb)模型的方式有三个不同之处。

1. 序列数据的不同采样方法（随机采样和顺序分区）将导致隐状态初始化的差异。
1. 我们在更新模型参数之前裁剪梯度。
   这样的操作的目的是，即使训练过程中某个点上发生了梯度爆炸，也能保证模型不会发散。
1. 我们用困惑度来评价模型。如 8.4.4节所述，
   这样的度量确保了不同长度的序列具有可比性。

具体来说，当使用顺序分区时，我们只在每个迭代周期的开始位置初始化隐状态。由于下一个小批量数据中的第$i$个子序列样本与当前第$i$个子序列样本相邻，因此当前小批量数据最后一个样本的隐状态，将用于初始化下一个小批量数据第一个样本的隐状态。这样，存储在隐状态中的序列的历史信息可以在一个迭代周期内流经相邻的子序列。然而，在任何一点隐状态的计算，都依赖于同一迭代周期中前面所有的小批量数据，这使得梯度计算变得复杂。为了降低计算量，在处理任何一个小批量数据之前，我们先分离梯度，使得隐状态的梯度计算总是限制在一个小批量数据的时间步内。

当使用随机抽样时，因为每个样本都是在一个随机位置抽样的，因此需要为每个迭代周期重新初始化隐状态。与 [3.6节](../03_pypto_linear_networks/03.06_softmax_regression_scratch.ipynb)中的`train_epoch_ch3`函数相同，`updater`是更新模型参数的常用函数。它既可以是从头开始实现的`d2l.sgd`函数，也可以是深度学习框架中内置的优化函数。


In [13]:
import math
from src.utils import Timer, Accumulator, sgd

def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期"""
    state, timer = None, Timer()
    metric = Accumulator(2)
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state = state.detach()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long())
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()

def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False, use_plot=True, loss_fn=None):
    """训练 RNN 
    Args:
        loss_fn: 可选，自定义损失函数（签名 loss(logits, labels) → 标量）
                 默认 nn.CrossEntropyLoss()
    """
    loss = loss_fn if loss_fn is not None else nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(predict("time traveller"))
            print(f"  [epoch {epoch+1}/{num_epochs}] perplexity={ppl:.1f}")
    print(f"困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}")
    print(predict("time traveller"))
    print(predict("traveller"))


训练流程与 [4.2 节](../04_pypto_multilayer_perceptrons/04.02_mlp_scratch.ipynb)基本一致，但 RNN 模型有三个特殊之处：
1. **隐状态初始化**：每批数据需调用 `net.begin_state()` 初始化隐状态
2. **标签展平**：RNN 输出 shape 为 `(batch*steps, vocab)`，标签需从 `(batch, steps)` 展平为 `(batch*steps,)`
3. **梯度裁剪**：更新参数前先裁剪梯度，防止长序列反向传播导致梯度爆炸

`loss.backward()` 时 PyTorch 沿计算图依次调用各 `autograd.Function.backward()`，触发对应的 pypto 反向 kernel。**首次运行**会触发编译（约 6 分钟），编译完成后直接复用缓存。


In [14]:
# 预热：触发所有 PyPTO kernel 的首次编译（首次约 6 分钟）
print('正在编译 pypto kernel（首次运行需约 6 分钟，请耐心等待）...')
import time; t0 = time.time()
# 参数初始化在 CPU 上，需移到 NPU
for p in net.params:
    p.data = p.data.to(device)
# 取一个批次，构造完整前向+反向计算来触发 JIT 编译
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = net.begin_state(X.shape[0], device)  # RNN 需要初始隐状态
y_hat, state = net(X, state)
y = y.T.reshape(-1).to(device)               # 标签展平为 (batch*steps,)
l = loss_fn(y_hat, y.long(), num_classes=len(vocab))
l.backward()
# 重置梯度，为正式训练做准备
for p in net.params:
    if p.grad is not None:
        p.grad = None
print(f'编译完成，耗时 {time.time()-t0:.0f} 秒。接下来可以正常训练了。')

正在编译 pypto kernel（首次运行需约 6 分钟，请耐心等待）...


编译完成，耗时 52 秒。接下来可以正常训练了。


In [15]:
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

time traveller the the the the the the the the the the the the t
  [epoch 10/500] perplexity=13.6


time traveller the the the the the the the the the the the the t
  [epoch 20/500] perplexity=10.7


time traveller the the the the the the the the the the the the t
  [epoch 30/500] perplexity=9.6


time travellere the the the the the the the the the the the the 
  [epoch 40/500] perplexity=9.1


time traveller and the the the the the the the the the the the t
  [epoch 50/500] perplexity=8.6


...

time traveller after the pauserequired for the proper assimilati
  [epoch 460/500] perplexity=1.2


time traveller ffreeedit espeecent simed indoul an theredither  
  [epoch 470/500] perplexity=1.2


time traveller sat nows ght on thethele wxoncubo the eit yoo sog
  [epoch 480/500] perplexity=1.1


time travelleryou can show black is white by argument said filby
  [epoch 490/500] perplexity=1.1


time traveller after the pauserequired for the proper assimilati
  [epoch 500/500] perplexity=1.1
困惑度 1.1, 12856.7 词元/秒 npu:0
time traveller after the pauserequired for the proper assimilati
traveller with a slight accession ofcheerfulness really thi


**最后，让我们检查一下使用随机抽样方法的结果。** 顺序采样中隐状态会在迭代之间传递，而随机抽样需要为每个小批量重新初始化隐状态。


In [16]:
net_random = PyPTORNNModelScratch(len(vocab), num_hiddens, device)
train_ch8(net_random, train_iter, vocab, lr, num_epochs, device, use_random_iter=True, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

time travellere the the the the the the the the the the the the 
  [epoch 10/500] perplexity=13.7


time travellere the the the the the the the the the the the the 
  [epoch 20/500] perplexity=10.6


time traveller the the the the the the the the the the the the t
  [epoch 30/500] perplexity=9.7


time traveller the the the the the the the the the the the the t
  [epoch 40/500] perplexity=9.0


time travellere and and and and and and and and and and and and 
  [epoch 50/500] perplexity=8.7


...

time traveller proceeded anyreal body must have extension in fou
  [epoch 460/500] perplexity=1.4


time travellerit s against reason said filby of course a solid b
  [epoch 470/500] perplexity=1.5


time travellerit s against reasot said filbycon thoume ias no rh
  [epoch 480/500] perplexity=1.6


time travellerit s against reason said the time travellerit s ag
  [epoch 490/500] perplexity=1.5


time travellerit s against reason said filbywhat le s dover man 
  [epoch 500/500] perplexity=1.4
困惑度 1.4, 13776.6 词元/秒 npu:0
time travellerit s against reason said filbywhat le s dover man 
travelleryou can show black is white by argument said filby


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期（定义见第8章）"""
    state, timer = None, d2l.Timer()
    metric = d2l.Accumulator(2)  # 训练损失之和,词元数量
    for X, Y in train_iter:
        if state is None or use_random_iter:
            # 在第一次迭代或使用随机抽样时初始化state
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                # state对于nn.GRU是个张量
                state.detach_()
            else:
                # state对于nn.LSTM或对于我们从零开始实现的模型是个张量
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            # 因为已经调用了mean函数
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False):
    """训练模型（定义见第8章）"""
    loss = nn.CrossEntropyLoss()
    animator = d2l.Animator(xlabel='epoch', ylabel='perplexity',
                            legend=['train'], xlim=[10, num_epochs])
    # 初始化
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: d2l.sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    # 训练和预测
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(predict('time traveller'))
            animator.add(epoch + 1, [ppl])
    print(f'困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}')
    print(predict('time traveller'))
    print(predict('traveller'))
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, d2l.try_gpu())
</pre>
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
net = RNNModelScratch(len(vocab), num_hiddens, d2l.try_gpu(), get_params,
                      init_rnn_state, rnn)
train_ch8(net, train_iter, vocab, lr, num_epochs, d2l.try_gpu(),
          use_random_iter=True)
</pre>
  </div>
</details>


## 8.5.7.小结

* 我们可以训练一个基于循环神经网络的字符级语言模型，根据用户提供的文本的前缀生成后续文本。

* 一个简单的循环神经网络语言模型包括输入编码、循环神经网络模型和输出生成。

* 循环神经网络模型在训练以前需要初始化状态，不过随机抽样和顺序划分使用初始化方法不同。

* 当使用顺序划分时，我们需要分离梯度以减少计算量。

* 在进行任何预测之前，模型通过预热期进行自我更新（例如，获得比初始值更好的隐状态）。

* 梯度裁剪可以防止梯度爆炸，但不能应对梯度消失。

## 8.5.8.练习

1. 尝试说明独热编码等价于为每个对象选择不同的嵌入表示。

1. 通过调整超参数（如迭代周期数、隐藏单元数、小批量数据的时间步数、学习率等）来改善困惑度。

    * 困惑度可以降到多少？

    * 用可学习的嵌入表示替换独热编码，是否会带来更好的表现？

    * 如果用H.G.Wells的其他书作为数据集时效果如何，

      例如[*世界大战*](http://www.gutenberg.org/ebooks/36)？

1. 修改预测函数，例如使用采样，而不是选择最有可能的下一个字符。

    * 会发生什么？

    * 调整模型使之偏向更可能的输出，例如，当$\alpha > 1$，从$q(x_t \mid x_{t-1}, \ldots, x_1) \propto P(x_t \mid x_{t-1}, \ldots, x_1)^\alpha$中采样。

1. 在不裁剪梯度的情况下运行本节中的代码会发生什么？

1. 更改顺序划分，使其不会从计算图中分离隐状态。运行时间会有变化吗？困惑度呢？

1. 用ReLU替换本节中使用的激活函数，并重复本节中的实验。我们还需要梯度裁剪吗？为什么？

参考答案详见 [answers/08.05_reference_answer](./answers/08.05_reference_answer.ipynb)。


### 参考答案（PyPTO）

In [ ]:
!cat answers/txt/08.05_reference_answer_pypto.txt

### 参考答案（PyTorch）

In [ ]:
!cat answers/txt/08.05_reference_answer_pytorch.txt